# E7 Notebook 29 — G2 Source-Integrity and Timestamp Forensics

**New protocol; not model training.** Our eight-file G1 is finished. The Thursday WebAttacks CSV contains a contiguous **288,602-row missing-label tail** (source rows 170,367–458,968) and original G0 recorded the same number of invalid timestamps and absent endpoints. Whether the tail is fully empty or corrupted with remaining features is **not yet established**. Its length also happens to equal the separate Thursday Afternoon Infiltration CSV; this is an observed count, **not evidence of why the anomaly occurred**.

G1 also found **208,847 Monday, 159,424 Tuesday, and 163,908 Wednesday original-row timestamp reversals**. We must determine which descending pairs are nominal 12:xx → 01:xx wrap candidates, and which are other order reversals. Neither establishes correct clock time or related-flow chronology.

**What Notebook 29 does:** 4 source-specific **full-file scans** (Thursday Morning WebAttacks structural nonempty fields; Monday, Tuesday, Wednesday time-order types). Produces and **immutably uploads one safe aggregate report after each completed CSV** to \`research/E7_CICIDS2017/results/G2_SOURCE_FORENSICS/\`. Existing G0/G1, original dataset bytes, prior models and frozen UNSW official test remain untouched. No field values, IPs, flow IDs or example rows are published. Reports can be recovered and verified if Kaggle stops.

**Requirements:** existing Kaggle dataset \`dsshayan/dscicids\`, Internet ON, secret \`pushRare\`. Run all cells. Source imports are version/commit/byte-hash pinned. There are no Git merge/pull/reset operations.

**Important limitations:** Original CIC-IDS2017 source provenance/corrected labels, twelve-hour AM/PM convention, true flow start/end, and leakage-free chronological grouping remain open research gates. No training or performance claims are authorized.

## 1. Verify version-pinned G2 forensics and publisher scripts plus original timestamp registry

In [ ]:
from pathlib import Path
import hashlib,importlib.util,json
from urllib.request import urlopen

REPO='itsCodeBakery/rare-attack-spatiotemporal-ids'
PINNED_COMMIT='0e265b7cf709d6de16ef13fea9ea12f464809cb2'
G2_REL='research/E7_CICIDS2017'
BLOBS={
  'scripts/e7_g2_source_clock_forensics.py':'517a3be65ffff4fa6a09e75e07020faf98f81141',
  'scripts/e7_g2_publish.py':'f01442d25b9f9a4f9c7438e3d794c641b8acec5c',
  'configs/e7_g0_timestamp_registry_001.json':'3e1c5f2e3252fa6923108eba02ac238b5a8561dc'
}
SNAPSHOT=Path('/kaggle/working/E7_G2_FORENSICS_PINNED_001')
SNAPSHOT.mkdir(parents=True,exist_ok=True)

def git_blob_sha(data):
    return hashlib.sha1(b'blob '+str(len(data)).encode()+bytes([0])+data).hexdigest()

for relative,expected in BLOBS.items():
    target=SNAPSHOT/relative
    if target.exists():
        data=target.read_bytes()
    else:
        url='https://raw.githubusercontent.com/'+REPO+'/'+PINNED_COMMIT+'/'+G2_REL+'/'+relative
        with urlopen(url,timeout=90) as response:data=response.read()
    assert git_blob_sha(data)==expected,'G2 source changed; refuse to run unverified code: '+relative
    if not target.exists():
        target.parent.mkdir(parents=True,exist_ok=True)
        target.write_bytes(data)

def load(name,path):
    spec=importlib.util.spec_from_file_location(name,path)
    module=importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

forensics=load('e7_g2_forensics',SNAPSHOT/'scripts/e7_g2_source_clock_forensics.py')
publisher=load('e7_g2_publisher',SNAPSHOT/'scripts/e7_g2_publish.py')
REGISTRY=json.loads((SNAPSHOT/'configs/e7_g0_timestamp_registry_001.json').read_text())
assert len(REGISTRY['filename_registry'])==8
print('SOURCE VERIFIED: 3 original pinned G2 scripts/configs')
print('PINNED COMMIT:',PINNED_COMMIT)
print('NO Git operations, model training, original data modifications or prior experiment updates.')

## 2. Verify four target CSV originals against previously completed G0 and G1 reports

In [ ]:
import base64
from urllib.parse import quote
from urllib.error import HTTPError
from kaggle_secrets import UserSecretsClient

SOURCE_DIR=Path('/kaggle/input/datasets/dsshayan/dscicids')
all_csv=sorted(SOURCE_DIR.rglob('*.csv'))
assert len(all_csv)==8 and {p.name for p in all_csv}==set(REGISTRY['filename_registry'])
assert all_csv[0].parent.name=='TrafficLabelling '
paths={p.name:p for p in all_csv}
SOURCES=[
  'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv',
  'Monday-WorkingHours.pcap_ISCX.csv',
  'Tuesday-WorkingHours.pcap_ISCX.csv',
  'Wednesday-workingHours.pcap_ISCX.csv'
]
token=UserSecretsClient().get_secret('pushRare')
assert token,'Set Kaggle secret pushRare; never paste the token into notebook outputs.'
publisher.api('GET','https://api.github.com/repos/'+REPO,token)

BASE='https://api.github.com/repos/'+REPO+'/contents/'
def get_saved(relative):
    obj=publisher.api('GET',BASE+quote(relative,safe='/'),token)
    return json.loads(base64.b64decode(obj['content']))

G0={};G1={};ENCODING={}
for name in SOURCES:
    old=get_saved(G2_REL+'/results/G0/'+name+'.audit.json')
    g1=get_saved(G2_REL+'/results/G1_LABEL_CLOCK/'+name+'.g1.json')
    sha=forensics.sha256_file(paths[name])
    assert sha==old['input_artifact_sha256']==g1['original_source_sha256']
    assert old['total_rows_inspected']==g1['rows']
    assert old['timestamp_format_declared']==REGISTRY['filename_registry'][name]['timestamp_format']
    encoding='cp1252' if any('using cp1252' in s for s in old['explicit_followup_gates']) else 'utf-8-sig'
    assert encoding==g1['source_text_encoding']
    G0[name]=old;G1[name]=g1;ENCODING[name]=encoding
    print('ORIGINAL SOURCE AUTHENTICATED:',name,'rows:',g1['rows'],'encoding:',encoding)

print('PREFLIGHT PASSED: four original source SHA256 hashes and corresponding G0/G1 reports match.')

## 3. Four full-file forensic scans; verified GitHub backup after each one

First checks whether the suspicious missing-label tail is **fully empty**, contains other traffic features, or mixes both. Remaining three checks distinguish nominal 12-to-1 clock drops from unrelated reversals, **without correcting times**. Existing G2 reports are reused when identical and source-checksum valid. Uploads go only to the G2 folder and are create-only; the publisher retains commit/blob receipts for a post-upload GitHub 404.

In [ ]:
ARTIFACTS=Path('/kaggle/working/E7_G2_SOURCE_FORENSICS_001')
ARTIFACTS.mkdir(parents=True,exist_ok=True)
results=[]

for i,name in enumerate(SOURCES,1):
    path=paths[name];spec=REGISTRY['filename_registry'][name]
    print('\nG2 SOURCE FORENSICS ['+str(i)+'/4] '+name,flush=True)
    dest=ARTIFACTS/(name+'.g2.json')

    if dest.exists():
        record=json.loads(dest.read_text(encoding='utf-8'))
        publisher.validate(record)
        assert record['source_sha256']==G0[name]['input_artifact_sha256']
        assert record['rows']==G0[name]['total_rows_inspected']
        print('Recovered original source-hash-verified G2 checkpoint.',flush=True)
    else:
        # Recover identical committed evidence if an earlier Kaggle session ended.
        url=BASE+quote(publisher.PREFIX+'/'+name+'.g2.json',safe='/')
        try:
            existing=publisher.api('GET',url,token)
        except HTTPError as e:
            if e.code!=404:raise
            existing=None
        if existing is not None:
            record=json.loads(base64.b64decode(existing['content']))
            publisher.validate(record)
            print('Recovered already committed immutable GitHub G2 report.',flush=True)
        elif name.startswith('Thursday'):
            record=forensics.missing_block(path,encoding=ENCODING[name],chunk_size=20000)
            print('Completed full original source structural-padding scan.',flush=True)
        else:
            record=forensics.clock_order(
                path,encoding=ENCODING[name],timestamp_format=spec['timestamp_format'],
                expected_weekday=spec['expected_weekday'],chunk_size=40000)
            print('Completed full original timestamp-order decomposition.',flush=True)
        publisher.validate(record)

    assert record['source_sha256']==G0[name]['input_artifact_sha256']
    assert record['rows']==G0[name]['total_rows_inspected']
    if name.startswith('Thursday'):
        assert record['counts']['no_label']==G1[name]['missing_label_rows']
        assert record['first_missing_label_row_1based']==G1[name]['first_missing_row_1based']
        assert record['last_missing_label_row_1based']==G1[name]['last_missing_row_1based']
        print('SOURCE MISSING LABELS:',record['counts']['no_label'],
              '| FULLY EMPTY SOURCE ROWS:',record['counts']['all_fields_empty'],
              '| MISSING LABEL BUT OTHER FIELDS PRESENT:',
              record['counts']['missing_label_but_other_fields_present'])
    else:
        assert record['adjacent_reversals_total']==G1[name]['source_adjacent_time_reversals'],(
            'G1 reversal count differs: stop and preserve original evidence')
        print('SOURCE TIME REVERSALS:',record['adjacent_reversals_total'],
              '| 12→1 reversals:',record['adjacent_12_to_1_hour_reversals'],
              '| other reversals:',record['adjacent_other_reversals'])
    verified=publisher.publish(record,token=token,local_dir=ARTIFACTS)
    print('GITHUB BACKUP:',verified['status'],verified.get('via',''),flush=True)
    print('PUBLIC REPORT: https://github.com/'+REPO+'/blob/main/'+verified['remote'])
    results.append(record)

print('\n'+'='*68)
print('E7 G2 COMPLETED: FOUR OF FOUR SOURCE FORENSIC REPORTS VERIFIED ON GITHUB')
print('PUBLIC DIRECTORY: https://github.com/'+REPO+'/tree/main/'+publisher.PREFIX)
print('Original G0/G1 and frozen UNSW results preserved. No training performed.')
print('Send the complete output to ChatGPT for G2 interpretation and G3 planning.')
print('='*68)

## G2 stop gate

These aggregate tests distinguish structurally absent original rows from rows that truly exist but lack labels, and quantify reversals inconsistent with chronological CSV order. **They do not establish the origin of the defect, recover labels, prove true timestamps or authorize temporal model fitting.** The follow-up will document source attestation, corrected CIC-IDS2017 provenance, clock conventions and a leakage-aware split design before training.